## 深入浅出pandas-6

我们再来看看`Index`类型，它为`Series`和`DataFrame`对象提供了索引服务，有了索引我们就可以排序数据（`sort_index`方法）、对齐数据（在运算和合并数据时非常重要）并实现对数据的快速检索（索引运算）。由于`DataFrame`类型表示的是二维数据，所以它的行和列都有索引，分别是`index`和`columns`。`Index`类型的创建的比较简单，通常给出`data`、`dtype`和`name`三个参数即可，分别表示作为索引的数据、索引的数据类型和索引的名称。由于`Index`本身也是一维的数据，索引它的方法和属性跟`Series`非常类似，你可以尝试创建一个`Index`对象，然后尝试一下之前学过的属性和方法在`Index`类型上是否生效。接下来，我们主要看看`Index`的几种子类型。

### 范围索引

范围索引是由具有单调性的整数构成的索引，我们可以通过`RangeIndex`构造器来创建范围索引，也可以通过`RangeIndex`类的类方法`from_range`来创建范围索引，代码如下所示。

代码：

In [ ]:
sales_data = np.random.randint(400, 1000, 12)
index = pd.RangeIndex(1, 13, name='月份')
ser = pd.Series(data=sales_data, index=index)
ser

### 分类索引

分类索引是由定类尺度构成的索引。如果我们需要通过索引将数据分组，然后再进行聚合操作，分类索引就可以派上用场。分类索引还有一个名为`reorder_categories`的方法，可以给索引指定一个顺序，分组聚合的结果会按照这个指定的顺序进行呈现，代码如下所示。

代码：

In [ ]:
sales_data = [6, 6, 7, 6, 8, 6]
index = pd.CategoricalIndex(
    data=['苹果', '香蕉', '苹果', '苹果', '桃子', '香蕉'],
    categories=['苹果', '香蕉', '桃子'],
    ordered=True
)
ser = pd.Series(data=sales_data, index=index)
ser

基于索引分组数据，然后使用`sum`进行求和。

In [ ]:
ser.groupby(level=0).sum()

指定索引的顺序。

In [ ]:
ser.index = index.reorder_categories(['香蕉', '桃子', '苹果'])
ser.groupby(level=0).sum()

### 多级索引

Pandas 中的`MultiIndex`类型用来表示层次或多级索引。可以使用`MultiIndex`类的类方法`from_arrays`、`from_product`、`from_tuples`等来创建多级索引，我们给大家举几个例子。

代码：

In [ ]:
tuples = [(1, 'red'), (1, 'blue'), (2, 'red'), (2, 'blue')]
index = pd.MultiIndex.from_tuples(tuples, names=['no', 'color'])
index

代码：

In [ ]:
arrays = [[1, 1, 2, 2], ['red', 'blue', 'red', 'blue']]
index = pd.MultiIndex.from_arrays(arrays, names=['no', 'color'])
index

代码：

In [ ]:
sales_data = np.random.randint(1, 100, 4)
ser = pd.Series(data=sales_data, index=index)
ser

代码：

In [ ]:
ser.groupby('no').sum()

代码：

In [ ]:
ser.groupby(level=1).sum()

代码：

In [ ]:
stu_ids = np.arange(1001, 1006)
semisters = ['期中', '期末']
index = pd.MultiIndex.from_product((stu_ids, semisters), names=['学号', '学期'])
courses = ['语文', '数学', '英语']
scores = np.random.randint(60, 101, (10, 3))
df = pd.DataFrame(data=scores, columns=courses, index=index)
df

根据第一级索引分组数据，按照期中成绩占`25%`，期末成绩占`75%` 的方式计算每个学生每门课的成绩。

代码：

In [ ]:
df.groupby(level=0).agg(lambda x: x.values[0] * 0.25 + x.values[1] * 0.75)

### 间隔索引

间隔索引顾名思义是使用固定的间隔范围充当索引，我们通常会使用`interval_range`函数来创建间隔索引，代码如下所示。

代码：

In [ ]:
index = pd.interval_range(start=0, end=5)
index

`IntervalIndex`有一个名为`contains`的方法，可以检查范围内是否包含了某个元素，如下所示。

代码：

In [ ]:
index.contains(1.5)

`IntervalIndex`还有一个名为`overlaps`的方法，可以检查一个范围跟其他的范围是否有重叠，如下所示。

代码：

In [ ]:
index.overlaps(pd.Interval(1.5, 3.5))

如果希望间隔范围是左闭右开的状态，可以在创建间隔索引时通过`closed='left'`来做到；如果希望两边都是关闭状态，可以将`close`参数的值赋值为`both`，代码如下所示。

代码：

In [ ]:
index = pd.interval_range(start=0, end=5, closed='left')
index

代码：

In [ ]:
index = pd.interval_range(start=pd.Timestamp('2022-01-01'), end=pd.Timestamp('2022-01-04'), closed='both')
index

### 日期时间索引

`DatetimeIndex`应该是众多索引中最复杂最重要的一种索引，我们通常会使用`date_range()`函数来创建日期时间索引，该函数有几个非常重要的参数`start`、`end`、`periods`、`freq`、`tz`，分别代表起始日期时间、结束日期时间、生成周期、采样频率和时区。我们先来看看如何创建`DatetimeIndex`对象，再来讨论它的相关运算和操作，代码如下所示。

代码：

In [ ]:
pd.date_range('2021-1-1', '2021-6-30', periods=10)

代码：

In [ ]:
pd.date_range('2021-1-1', '2021-6-30', freq='W')

> **说明**：`freq=W`表示采样周期为一周，它会默认星期日是一周的开始；如果你希望星期一表示一周的开始，你可以将其修改为`freq=W-MON`；你也可以试着将该参数的值修改为`12H`，`M`，`Q`等，看看会发生什么，相信你不难猜到它们的含义。

`DatatimeIndex`可以跟`DateOffset`类型进行运算，这一点很好理解，以为我们可以设置一个时间差让时间向前或向后偏移，具体的操作如下所示。

代码：

In [ ]:
index = pd.date_range('2021-1-1', '2021-6-30', freq='W')
index - pd.DateOffset(days=2)

代码：

In [ ]:
index + pd.DateOffset(hours=2, minutes=10)

如果`Series`对象或`DataFrame`对象使用了`DatetimeIndex`类型的索引，此时我们可以通过`asfreq()`方法指定一个时间频率来实现对数据的抽样，我们仍然以之前讲过的百度股票数据为例，给大家做一个演示。

代码：

In [ ]:
baidu_df = pd.read_excel('data/2022年股票数据.xlsx', sheet_name='BIDU', index_col='Date')
baidu_df.sort_index(inplace=True)
baidu_df.asfreq('5D')

大家可能注意到了，每5天抽取1天有可能会抽中非交易日，那么对应的列都变成了空值，为了解决这个问题，在使用`asfreq`方法时可以通过`method`参数来指定一种填充空值的方法，可以将相邻的交易日的数据填入进来。

代码：

In [ ]:
baidu_df.asfreq('5D', method='ffill')

当使用`DatetimeIndex`索引时，我们也可以通过`resample()`方法基于时间对数据进行重采样，相当于根据时间周期对数据进行了分组操作，分组之后还可以进行聚合统计，代码如下所示。

代码：

In [ ]:
baidu_df.resample('1M').mean()

代码：

In [ ]:
baidu_df.resample('1M').agg(['mean', 'std'])

> **提示**：不知大家是否注意到，上面输出的`DataFrame` 的列索引是一个`MultiIndex`对象。你可以访问上面的`DataFrame`对象的`columns`属性看看。

如果要实现日期时间的时区转换，我们可以先用`tz_localize()`方法将日期时间本地化，代码如下所示。

代码：

In [ ]:
baidu_df = baidu_df.tz_localize('Asia/Chongqing')
baidu_df

在对时间本地化以后，我们再使用`tz_convert()`方法就可以实现转换时区，代码如下所示。

代码：

In [ ]:
baidu_df.tz_convert('America/New_York')

如果你的数据使用了`DatetimeIndex`类型的索引，那么你就很有可能要对数据进行时间序列分析，关于时间序列分析的方法和模型并不是本章节要探讨的内容，我们在其他的专栏中为大家分享。